In [9]:
# Filename......: L11_OlisBahari_ITAI1371
# Language......: Python
# Tools.........: Visual Studio Code (VSC)
#               : Google Colab
# Class.........: ITAI 1371 Introduction to Machine Learning
# Semester......: Summer 2026
# Class Type....: Online
# Instructor....: Sitaram Ayyagari
# Student.......: Olis Bahari
# Version.......: V1.0
# Purpose.......: 

In [10]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [11]:
def Random_Forest_Model_Routine():
    iris = load_iris()
    X = iris.data
    y = iris.target

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

    # A baseline model with default hyperparameters
    baseline_model = RandomForestClassifier(random_state=42)
    baseline_model.fit(X_train, y_train)
    y_pred_baseline = baseline_model.predict(X_test)
    accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

    print("\n" + "=" * 60)
    print("Random Forest Baseline Model")

    print(f"Accuracy Random Forest: {accuracy_baseline:.2%}")

    return   X_train, X_test, y_train, y_test, accuracy_baseline, iris

In [12]:
def Grid_Search_Model_Routine(X_train, X_test, y_train, y_test):
    # Define all hyperparameter values to test
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, None]
    }

    # Grid Search tests every possible combination
    grid_search = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2
    )

    # Train all parameter combinations
    grid_search.fit(X_train, y_train)

    # Use the best model found by Grid Search
    best_grid_model = grid_search.best_estimator_

    # Make predictions on the test data
    y_pred_grid = best_grid_model.predict(X_test)

    # Calculate test accuracy
    accuracy_grid = accuracy_score(
        y_test,
        y_pred_grid
    )

    print("\n" + "=" * 60)
    print("Grid Search Result")
    print(f"Best parameters: {grid_search.best_params_}")
    print(
        f"Best cross-validated score: "
        f"{grid_search.best_score_:.2%}"
    )
    print(f"Grid Search test accuracy: {accuracy_grid:.2%}")

    return accuracy_grid

In [13]:
def Randomized_Search_Model_Routine():
    # Define a larger collection of parameter values
    param_dist = {
        "n_estimators": [
            int(x)
            for x in np.linspace(
                start=50,
                stop=500,
                num=10
            )
        ],
        "max_depth": [5, 10, 20, 30, None],
        "min_samples_split": [2, 5, 10]
    }

    # Randomized Search tests only 10 randomly selected combinations
    random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=10,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2,
        random_state=42
    )

    # Train the randomly selected combinations
    random_search.fit(X_train, y_train)

    # Use the best model found by Randomized Search
    best_random_model = random_search.best_estimator_

    # Make predictions on the test data
    y_pred_random = best_random_model.predict(X_test)

    # Calculate test accuracy
    accuracy_random = accuracy_score(
        y_test,
        y_pred_random
    )

    print("\n" + "=" * 60)
    print("Randomized Search Result")
    print(f"Best parameters: {random_search.best_params_}")
    print(
        f"Best cross-validated score: "
        f"{random_search.best_score_:.2%}"
    )
    print(
        f"Randomized Search test accuracy: "
        f"{accuracy_random:.2%}"
    )

    return accuracy_random

In [14]:
def AutoGluon_Model_Routine(iris):

    try:
        from autogluon.tabular import TabularPredictor

        # AutoGluon requires a DataFrame containing both
        # the features and target column.
        train_data_ag = pd.DataFrame(
            X_train,
            columns=iris.feature_names
        )

        train_data_ag["species"] = y_train

        test_data_ag = pd.DataFrame(
            X_test,
            columns=iris.feature_names
        )

        test_data_ag["species"] = y_test

        # Create and train the AutoGluon predictor
        predictor = TabularPredictor(
            label="species",
            eval_metric="accuracy"
        ).fit(
            train_data=train_data_ag,
            time_limit=60
        )

        # Display model performance
        leaderboard = predictor.leaderboard(
            test_data_ag,
            silent=True
        )

        print("\n" + "=" * 60)
        print("Autogluon Leaderboard")


        # Evaluate the best AutoGluon model
        autogluon_results = predictor.evaluate(test_data_ag)

        print("\nAutoGluon evaluation:")
        print(autogluon_results)

    except ImportError:
        print("\n" + "=" * 60)
        print("AUTOGLUON NOT INSTALLED")

In [15]:
def Model_Result_Routine(accuracy_baseline):

    model_results = {
        "Baseline Random Forest": accuracy_baseline,
        "Grid Search": accuracy_grid,
        "Randomized Search": accuracy_random
    }

    print("\n" + "=" * 60)
    print("Final Model Comparison")


    for model_name, model_accuracy in model_results.items():
        print(f"{model_name:<30} {model_accuracy:.2%}")

    best_method = max(model_results, key=model_results.get)

    print(
        f"\nBest scikit-learn method: {best_method} "
        f"({model_results[best_method]:.2%})"
    )

In [16]:
# <<< Main >>>

X_train, X_test, y_train, y_test, accuracy_baseline, iris = Random_Forest_Model_Routine()

accuracy_grid=Grid_Search_Model_Routine(  X_train, X_test, y_train, y_test)

accuracy_random=Randomized_Search_Model_Routine()

AutoGluon_Model_Routine(iris)

Model_Result_Routine(accuracy_baseline)


Random Forest Baseline Model
Accuracy Random Forest: 88.89%
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Grid Search Result
Best parameters: {'max_depth': 5, 'n_estimators': 50}
Best cross-validated score: 95.24%
Grid Search test accuracy: 88.89%
Fitting 5 folds for each of 10 candidates, totalling 50 fits


No path specified. Models will be saved in: "AutogluonModels\ag-20260711_134420"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.8
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22621
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       15.37 GB / 31.75 GB (48.4%)
Disk Space Avail:   344.48 GB / 475.84 GB (72.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitr


Randomized Search Result
Best parameters: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 95.24%
Randomized Search test accuracy: 91.11%


Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "c:\Dropbox\Artifical Intellegence\Houston City College (HCC)\02 Introduction to Machine Learning\Olis-Bahari-ML-ITAI-1371\Labs\Lab-11\AutogluonModels\ag-20260711_134420"
Train Data Rows:    105
Train Data Columns: 4
Label Column:       species
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == int, but few unique label-values observed).
	3 unique label values:  [np.int64(1), np.int64(0), np.int64(2)]
	If 'multiclass' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       multiclass
Preprocessing data ...
Train Data Class Count: 3
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    15728.54 MB
	Train Data (Original)  Memory Usa


Autogluon Leaderboard

AutoGluon evaluation:
{'accuracy': 0.9111111111111111, 'balanced_accuracy': np.float64(0.9111111111111111), 'mcc': 0.8692460374493866}

Final Model Comparison
Baseline Random Forest         88.89%
Grid Search                    88.89%
Randomized Search              91.11%

Best scikit-learn method: Randomized Search (91.11%)
